In [1]:
import zipfile
import re
import rasterio
from rasterio.io import MemoryFile

sar_zip = "SN6_25_SAR.zip"
optical_zip = "SN6_25_Optical.zip"
mask_zip = "SN6_25_Masks.zip"

def list_members(zip_path):
    with zipfile.ZipFile(zip_path) as zf:
        return zf.namelist()

sar_names = [n for n in list_members(sar_zip) if n.endswith(".tif")]
optical_names = [n for n in list_members(optical_zip) if n.endswith(".tif")]
mask_names = [n for n in list_members(mask_zip) if n.endswith((".tif", ".geojson", ".png"))]  # check actual format

def get_tile_id(fname):
    m = re.search(r"tile_(\d+)", fname)
    return m.group(1) if m else None

# Build tile_id -> filename lookups per archive
sar_by_id = {get_tile_id(n): n for n in sar_names if get_tile_id(n)}
optical_by_id = {get_tile_id(n): n for n in optical_names if get_tile_id(n)}
mask_by_id = {get_tile_id(n): n for n in mask_names if get_tile_id(n)}

# Sanity check: do all three archives actually agree on which tiles exist?
common_ids = set(sar_by_id) & set(optical_by_id) & set(mask_by_id)
print(f"SAR: {len(sar_by_id)}, Optical: {len(optical_by_id)}, Mask: {len(mask_by_id)}")
print(f"Common to all three: {len(common_ids)}")

SAR: 850, Optical: 850, Mask: 850
Common to all three: 850


In [2]:
with zipfile.ZipFile(mask_zip) as zf:
    print(zf.namelist()[:5])

['SN6_Train_AOI_11_Rotterdam_SAR-Intensity_20190822162246_20190822162529_tile_7653.tif', 'SN6_Train_AOI_11_Rotterdam_SAR-Intensity_20190804124148_20190804124417_tile_1326.tif', 'SN6_Train_AOI_11_Rotterdam_SAR-Intensity_20190823150417_20190823150712_tile_10311.tif', 'SN6_Train_AOI_11_Rotterdam_SAR-Intensity_20190822155902_20190822160145_tile_890.tif', 'SN6_Train_AOI_11_Rotterdam_SAR-Intensity_20190823144849_20190823145144_tile_3293.tif']


In [3]:
match = list(mask_by_id.values())[0]
if match.endswith((".tif", ".png")):
    arr, profile = read_tif_from_zip(mask_zip, match)
    print(arr.dtype, np.unique(arr), arr.shape)

NameError: name 'read_tif_from_zip' is not defined

In [4]:
print(sar_names[:3])
print(optical_names[:3])

['SN6_Train_AOI_11_Rotterdam_SAR-Intensity_20190822162246_20190822162529_tile_7653.tif', 'SN6_Train_AOI_11_Rotterdam_SAR-Intensity_20190804124148_20190804124417_tile_1326.tif', 'SN6_Train_AOI_11_Rotterdam_SAR-Intensity_20190823150417_20190823150712_tile_10311.tif']
['SN6_Train_AOI_11_Rotterdam_PS-RGB_20190822162246_20190822162529_tile_7653.tif', 'SN6_Train_AOI_11_Rotterdam_PS-RGB_20190804124148_20190804124417_tile_1326.tif', 'SN6_Train_AOI_11_Rotterdam_PS-RGB_20190823150417_20190823150712_tile_10311.tif']


In [5]:
with zipfile.ZipFile(mask_zip) as zf:
    names = zf.namelist()
print(names[:5])

['SN6_Train_AOI_11_Rotterdam_SAR-Intensity_20190822162246_20190822162529_tile_7653.tif', 'SN6_Train_AOI_11_Rotterdam_SAR-Intensity_20190804124148_20190804124417_tile_1326.tif', 'SN6_Train_AOI_11_Rotterdam_SAR-Intensity_20190823150417_20190823150712_tile_10311.tif', 'SN6_Train_AOI_11_Rotterdam_SAR-Intensity_20190822155902_20190822160145_tile_890.tif', 'SN6_Train_AOI_11_Rotterdam_SAR-Intensity_20190823144849_20190823145144_tile_3293.tif']


In [6]:
match = names[0]
arr, profile = read_tif_from_zip(mask_zip, match)
print(arr.dtype, np.unique(arr), arr.shape)

NameError: name 'read_tif_from_zip' is not defined

In [7]:
import zipfile
import numpy as np
import rasterio
from rasterio.io import MemoryFile

def read_tif_from_zip(zip_path, filename):
    """Read a single .tif file from a ZIP archive into a numpy array + profile."""
    with zipfile.ZipFile(zip_path) as zf:
        with zf.open(filename) as f:
            # Wrap the file-like object in a MemoryFile so rasterio can read it
            with MemoryFile(f.read()) as memfile:
                with memfile.open() as dataset:
                    arr = dataset.read()        # numpy array
                    profile = dataset.profile   # metadata
    return arr, profile

# Example usage
with zipfile.ZipFile(mask_zip) as zf:
    names = zf.namelist()
print(names[:5])

match = names[0]
arr, profile = read_tif_from_zip(mask_zip, match)
print(arr.dtype, np.unique(arr), arr.shape)


['SN6_Train_AOI_11_Rotterdam_SAR-Intensity_20190822162246_20190822162529_tile_7653.tif', 'SN6_Train_AOI_11_Rotterdam_SAR-Intensity_20190804124148_20190804124417_tile_1326.tif', 'SN6_Train_AOI_11_Rotterdam_SAR-Intensity_20190823150417_20190823150712_tile_10311.tif', 'SN6_Train_AOI_11_Rotterdam_SAR-Intensity_20190822155902_20190822160145_tile_890.tif', 'SN6_Train_AOI_11_Rotterdam_SAR-Intensity_20190823144849_20190823145144_tile_3293.tif']
uint8 [  0 255] (1, 900, 900)


In [8]:
def read_triple(tile_id):
    sar_arr, sar_profile = read_tif_from_zip(sar_zip, sar_by_id[tile_id])
    opt_arr, opt_profile = read_tif_from_zip(optical_zip, optical_by_id[tile_id])
    mask_arr, _ = read_tif_from_zip(mask_zip, mask_by_id[tile_id])
    mask_arr = (mask_arr > 0).astype(np.float32)   # {0,255} → {0.,1.}
    return sar_arr, sar_profile, opt_arr, opt_profile, mask_arr

In [12]:
import zipfile
import numpy as np
import rasterio
from rasterio.io import MemoryFile
import json

# --- Utility: read a single .tif from ZIP ---
def read_tif_from_zip(zip_path, filename):
    with zipfile.ZipFile(zip_path) as zf:
        with zf.open(filename) as f:
            with MemoryFile(f.read()) as memfile:
                with memfile.open() as dataset:
                    arr = dataset.read()
                    profile = dataset.profile
    return arr, profile

# --- Triple reader: SAR, Optical, Mask ---
def read_triple(tile_id, sar_zip, optical_zip, mask_zip,
                sar_by_id, optical_by_id, mask_by_id):
    sar_arr, sar_profile = read_tif_from_zip(sar_zip, sar_by_id[tile_id])
    opt_arr, opt_profile = read_tif_from_zip(optical_zip, optical_by_id[tile_id])
    mask_arr, _ = read_tif_from_zip(mask_zip, mask_by_id[tile_id])
    mask_arr = (mask_arr > 0).astype(np.float32)   # {0,255} → {0.,1.}
    return sar_arr, sar_profile, opt_arr, opt_profile, mask_arr

# --- Dimension check ---
def check_shapes(sample_id, sar_zip, optical_zip, mask_zip,
                 sar_by_id, optical_by_id, mask_by_id):
    sar_arr, sar_p, opt_arr, opt_p, mask_arr = read_triple(
        sample_id, sar_zip, optical_zip, mask_zip,
        sar_by_id, optical_by_id, mask_by_id
    )
    print("SAR:", sar_arr.shape, sar_arr.dtype)
    print("Optical:", opt_arr.shape, opt_arr.dtype)
    print("Mask:", mask_arr.shape, mask_arr.dtype)

# --- Radiometric statistics ---
def compute_stats(tile_ids, sar_zip, optical_zip,
                  sar_by_id, optical_by_id):
    sar_vals = []
    opt_vals = []
    for tid in tile_ids:
        sar_arr, _ = read_tif_from_zip(sar_zip, sar_by_id[tid])
        opt_arr, _ = read_tif_from_zip(optical_zip, optical_by_id[tid])

        # SAR: convert to dB (avoid log(0))
        sar_db = 10 * np.log10(np.clip(sar_arr.astype(np.float32), 1e-6, None))
        sar_vals.append(sar_db.flatten())

        # Optical: raw reflectance values
        opt_vals.append(opt_arr.astype(np.float32).flatten())

    sar_vals = np.concatenate(sar_vals)
    opt_vals = np.concatenate(opt_vals)

    stats = {
        "sar": {
            "mean": float(np.mean(sar_vals)),
            "std": float(np.std(sar_vals)),
            "p2": float(np.percentile(sar_vals, 2)),
            "p98": float(np.percentile(sar_vals, 98))
        },
        "optical": {
            "mean": float(np.mean(opt_vals)),
            "std": float(np.std(opt_vals)),
            "p2": float(np.percentile(opt_vals, 2)),
            "p98": float(np.percentile(opt_vals, 98))
        }
    }
    return stats

# --- Final config writer ---
def write_config(stats, out_path="preprocessing_config.json"):
    config = {
        "crop_size": [512, 512],
        "sar_stats": stats["sar"],
        "optical_stats": stats["optical"]
    }
    with open(out_path, "w") as f:
        json.dump(config, f, indent=2)
    print(f"Config written to {out_path}")

# --- Example run ---
sample_id = list(common_ids)[0]
check_shapes(sample_id, sar_zip, optical_zip, mask_zip,
             sar_by_id, optical_by_id, mask_by_id)
stats = compute_stats(train_ids, sar_zip, optical_zip,
                      sar_by_id, optical_by_id)
write_config(stats)


SAR: (4, 900, 900) float32
Optical: (4, 900, 900) uint8
Mask: (1, 900, 900) float32
Config written to preprocessing_config.json


In [11]:
from sklearn.model_selection import train_test_split

# common_ids should be a set or list of tile IDs that exist in all three sources
train_ids, val_ids = train_test_split(
    list(common_ids),
    test_size=0.2,
    random_state=42
)

print("Train size:", len(train_ids))
print("Val size:", len(val_ids))


Train size: 680
Val size: 170


In [14]:
import numpy as np

def compute_stats_per_channel(tile_ids, zip_path, by_id, is_sar=False):
    stack = []
    for tid in tile_ids:
        arr, _ = read_tif_from_zip(zip_path, by_id[tid])
        if is_sar:
            arr = 10 * np.log10(arr.astype(np.float32) + 1e-6)
        stack.append(arr.reshape(arr.shape[0], -1))  # (bands, pixels)

    stacked = np.concatenate(stack, axis=1)  # (bands, total_pixels)

    means = stacked.mean(axis=1).tolist()
    stds = stacked.std(axis=1).tolist()
    p2, p98 = np.percentile(stacked, [2, 98], axis=1)

    return {
        "mean": means,
        "std": stds,
        "p2": p2.tolist(),
        "p98": p98.tolist()
    }

In [15]:
sar_stats = compute_stats_per_channel(train_ids, sar_zip, sar_by_id, is_sar=True)
opt_stats = compute_stats_per_channel(train_ids, optical_zip, optical_by_id, is_sar=False)

config = {
    "crop_size": [512, 512],
    "splits": {
        "train_ids": train_ids,
        "val_ids": val_ids,
        "test_ids": test_ids
    },
    "sar_stats": sar_stats,
    "optical_stats": opt_stats
}

with open("preprocessing_config.json", "w") as f:
    json.dump(config, f, indent=2)

print("✅ preprocessing_config.json written")


✅ preprocessing_config.json written
